In [3]:
DATASET_PATH = "/kaggle/input/datasets/sachchitkunichetty/rvf10k/rvf10k"

In [4]:
import os

print(os.listdir(DATASET_PATH))
print(os.listdir(DATASET_PATH + "/train"))
print(os.listdir(DATASET_PATH + "/valid"))

['valid', 'train']
['fake', 'real']
['fake', 'real']


In [5]:
# ═══════════════════════════════════════════════════════════
#  FINAL RVF10K TRAINING (KAGGLE + FIXED PATH VERSION)
# ═══════════════════════════════════════════════════════════

import os, random, warnings
warnings.filterwarnings("ignore")

import torch
from PIL import Image, ImageEnhance
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, SiglipForImageClassification
from tqdm import tqdm

# ───────────────── CONFIG ─────────────────
MODEL_ID = "prithivMLmods/deepfake-detector-model-v1"

# ✅ FINAL CONFIRMED PATH
DATASET_PATH = "/kaggle/input/datasets/sachchitkunichetty/rvf10k/rvf10k"

EPOCHS = 2
BATCH_SIZE = 16
LR = 1e-5
MAX_SAMPLES = 3000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

# ── VERIFY PATH ──────────────────────────
print("Dataset:", os.listdir(DATASET_PATH))

# ── LOAD MODEL ───────────────────────────
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = SiglipForImageClassification.from_pretrained(MODEL_ID)
model.to(DEVICE)

# ✅ Freeze everything
for param in model.parameters():
    param.requires_grad = False

# ✅ Train only classifier
for param in model.classifier.parameters():
    param.requires_grad = True

print("Model ready\n")

# ── IMAGE LOADER ─────────────────────────
def load_images(folder, label, limit):
    data = []
    files = os.listdir(folder)[:limit]

    for f in files:
        try:
            img = Image.open(os.path.join(folder, f)).convert("RGB")
            data.append((img, label))
        except:
            continue

    return data

# ── LOAD DATA ────────────────────────────
train_real_path = os.path.join(DATASET_PATH, "train", "real")
train_fake_path = os.path.join(DATASET_PATH, "train", "fake")

val_real_path = os.path.join(DATASET_PATH, "valid", "real")
val_fake_path = os.path.join(DATASET_PATH, "valid", "fake")

print("Loading dataset...")

train_real = load_images(train_real_path, 1, MAX_SAMPLES // 2)
train_fake = load_images(train_fake_path, 0, MAX_SAMPLES // 2)

val_real = load_images(val_real_path, 1, MAX_SAMPLES // 10)
val_fake = load_images(val_fake_path, 0, MAX_SAMPLES // 10)

train_samples = train_real + train_fake
val_samples = val_real + val_fake

random.shuffle(train_samples)
random.shuffle(val_samples)

print(f"Train: {len(train_samples)} | Val: {len(val_samples)}")

# ── DATASET CLASS ────────────────────────
class DeepfakeDataset(Dataset):
    def __init__(self, samples, augment=False):
        self.samples = samples
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image, label = self.samples[idx]

        image = image.resize((224, 224))

        if self.augment:
            if random.random() > 0.5:
                image = image.transpose(Image.FLIP_LEFT_RIGHT)

            image = ImageEnhance.Brightness(image).enhance(random.uniform(0.7, 1.3))
            image = ImageEnhance.Contrast(image).enhance(random.uniform(0.7, 1.3))

        inputs = processor(images=image, return_tensors="pt")

        return {
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "labels": torch.tensor(label)
        }

train_loader = DataLoader(
    DeepfakeDataset(train_samples, True),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    DeepfakeDataset(val_samples, False),
    batch_size=BATCH_SIZE
)

# ── OPTIMIZER ────────────────────────────
optimizer = torch.optim.AdamW(
    model.classifier.parameters(),
    lr=LR
)

loss_fn = torch.nn.CrossEntropyLoss()

# ── TRAINING ─────────────────────────────
best_val = 0

for epoch in range(EPOCHS):

    model.train()
    correct = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):

        pv = batch["pixel_values"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(pixel_values=pv)

        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        correct += (outputs.logits.argmax(1) == labels).sum().item()

    train_acc = correct / len(train_samples)

    # ── VALIDATION ──
    model.eval()
    correct = 0

    with torch.no_grad():
        for batch in val_loader:
            pv = batch["pixel_values"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            outputs = model(pixel_values=pv)
            correct += (outputs.logits.argmax(1) == labels).sum().item()

    val_acc = correct / len(val_samples)

    print(f"\nEpoch {epoch+1} | Train: {train_acc:.4f} | Val: {val_acc:.4f}")

    # ✅ Early stopping
    if val_acc > 0.95:
        print("Stopping early (good generalization)")
        break

# ── SAVE MODEL ───────────────────────────
model.save_pretrained("/kaggle/working/rvf10k_model")
processor.save_pretrained("/kaggle/working/rvf10k_model")

print("\nModel saved successfully ✅")

Device: cuda
Dataset: ['valid', 'train']


preprocessor_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/372M [00:00<?, ?B/s]

...


Epoch 1 | Train: 0.8387 | Val: 0.9167


Epoch 2: 100%|██████████| 188/188 [00:49<00:00,  3.82it/s]



Epoch 2 | Train: 0.8827 | Val: 0.9400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved successfully ✅
